## 依赖导入

本 notebook 涉及灰色预测 GM(1,1)、马尔可夫链平稳分布与状态转移、灰态/序列的频数统计，以及神经网络与感知器建模（sklearn 的 `Perceptron`、`MLPClassifier`、`MLPRegressor`）。统一引入 `matplotlib`、`numpy` 与 `sklearn` 相关模型。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier, MLPRegressor

## 全局绘图配置

用 `plt.rc` 设置全局字号 15、中文字体 SimHei，保证后续图中中文标签正常显示。

In [ ]:
plt.rc("font", size=15)
plt.rc("font", family="SimHei")

## 13.4  灰色预测 GM(1,1) 模型

对原始序列 x0 作一次累加得 x1，用紧邻均值生成 z，构造背景矩阵 $B=[-z, z^2]^T$，以最小二乘 `np.linalg.pinv(B)@x0[1:]` 拟合参数 u（发展系数 a、 grey 作用量 b）。再由解的闭式表达式定义 `x(t)`（累加预测序列），`np.diff` 还原得 xh0，计算预测残差 ea 与相对误差 er（%）。（源程序 ex15_4.py）

In [ ]:
x0 = np.array([4.93, 2.33, 3.87, 4.35, 6.63, 7.15, 5.37, 6.39, 7.81, 8.35])
n = len(x0)
x1 = np.cumsum(x0)
z = (x1[1:] + x1[:-1]) / 2
B = np.vstack([-z, z**2]).T
u = np.linalg.pinv(B) @ x0[1:]
print("参数 u：", np.round(u, 4))


def x(t):
    return u[0] * x0[0] / (u[1] * x0[0] + (u[0] - u[1] * x0[0]) * np.exp(u[0] * t))


xh1 = x(np.arange(n))
xh0 = np.hstack([x0[0], np.diff(xh1)])
ea = x0 - xh0
er = abs(ea) / x0 * 100

## 13.5  相邻状态转移频数统计（2 状态）

读取 `data/15_5.txt` 并去换行得到状态序列字符串 s。`mfind(s,c)` 返回子串 c 在 s 中所有（重叠）出现位置。遍历状态对 (i,j)，统计 “i 之后紧跟 j” 的出现次数填入转移频数矩阵 a，打印矩阵及各行之和。（`a.sum(axis=1)` 即行和）（源程序 ex15_5.py）

In [ ]:
with open("data/15_5.txt") as f:
    s = f.read().replace("\n", "")
a = np.zeros((2, 2))


def mfind(s, c):
    return [x for x in range(s.find(c), len(s)) if s[x : x + 2] == c]


for i in range(2):
    for j in range(2):
        a[i, j] = len(mfind(s, str(i) + str(j)))
print("统计数据矩阵 a:\n", a)
print("a 的行和：", a.sum(axis=1))

## 13.6  相邻状态转移频数统计（4 状态）

与上一格方法相同，但先去掉空格与换行，状态编号 1~4，构造 4×4 转移频数矩阵 a，打印矩阵与各行之和。（源程序 ex15_6.py）

In [ ]:
with open("data/15_6.txt") as f:
    s = f.read().replace(" ", "").replace("\n", "")
a = np.zeros((4, 4))


def mfind(s, c):
    return [x for x in range(s.find(c), len(s)) if s[x : x + 2] == c]


for i in range(1, 5):
    for j in range(1, 5):
        a[i - 1, j - 1] = len(mfind(s, str(i) + str(j)))
print("统计数据矩阵 a:\n", a)
print("a 的行和：", a.sum(axis=1))

## 13.7  马尔可夫链多步状态转移

初始分布 P1 与一步转移矩阵 P 均用 `np.asmatrix` 表示，计算三步后分布 `P4=P1@P**3` 并打印。（源程序 ex15_7.py）

In [ ]:
P1 = np.asmatrix([0.2, 0.4, 0.4])
P = np.asmatrix([[0.8, 0.1, 0.1], [0.5, 0.1, 0.4], [0.5, 0.3, 0.2]])
P4 = P1 @ P**3
print("P4:", P4)

## 13.8  马尔可夫链平稳分布

本节的 (1) 用线性方程组求解平稳分布（源程序 ex15_8_1.py）：把 πP=π 与 Σπ=1 联立，系数矩阵 `a=np.vstack([p.T-np.eye(3), np.ones(3)])`、常数项 `b=np.hstack([np.zeros(3),1])`，以 `np.linalg.pinv(a)@b` 求最小二乘数值解并保留 4 位打印。

In [ ]:
p = np.array([[0.8, 0.1, 0.1], [0.5, 0.1, 0.4], [0.5, 0.3, 0.2]])
a = np.vstack([p.T - np.eye(3), np.ones(3)])
b = np.hstack([np.zeros(3), 1])
x = np.linalg.pinv(a) @ b
print("解为：", np.round(x, 4))

本节的 (2) 改用特征值法（源程序 ex15_8_2.py）：对 `p.T` 作 `np.linalg.eig`，取最大特征值对应的特征向量 `vec[:,0]`，归一化（`vec[:,0]/vec[:,0].sum()`，`.sum()` 与原 `sum()` 等价，已断言一致）得平稳分布并打印。原先赋给 `val` 的特征值未被使用，改为 `_`。

In [ ]:
p = np.array([[0.8, 0.1, 0.1], [0.5, 0.1, 0.4], [0.5, 0.3, 0.2]])
_, vec = np.linalg.eig(p.T)
s = vec[:, 0] / vec[:, 0].sum()
print("求得特征向量为：", np.round(s, 4))

## 13.9  马尔可夫链平稳分布（另一转移矩阵）

方法与 13.8(1) 相同（线性方程组 + 最小二乘），仅转移矩阵 p 换为另一矩阵；此处把 Σπ=1 约束行写成 `np.ones((1,3))`。打印数值解（保留 4 位）。（源程序 ex15_9.py）

In [ ]:
p = np.array([[0.2, 0.8, 0], [0.8, 0, 0.2], [0.1, 0.3, 0.6]])
a = np.vstack([p.T - np.eye(3), np.ones((1, 3))])
b = np.hstack([np.zeros(3), 1])
x = np.linalg.pinv(a) @ b
print("解为：", np.round(x, 4))

## 13.10  感知器分类模型

构造二维训练样本 x0 与标签 y0，用 sklearn `Perceptron()` 拟合并打印其系数 coef_、截距 intercept_、在训练集上的 score 精度，以及对样本 [-0.5,0.2] 的预测值。（源程序 ex15_10.py）

In [ ]:
x0 = np.array([[-0.5, -0.5, 0.3, 0.0], [-0.5, 0.5, -0.5, 1.0]]).T
y0 = np.array([1, 1, 0, 0])
md = Perceptron().fit(x0, y0)
print("模型系数和常数项分别为：", md.coef_, ",", md.intercept_)
print("模型精度：", md.score(x0, y0))
print("预测值为：", md.predict([[-0.5, 0.2]]))

## 13.11  MLP 分类器判别样本类别

读取 `data/15_11.txt`，前 10 行为训练样本 x0、其余为待判样本 x。对训练样本逐列取最大值 m1/最小值 m2 做极差标准化，其中 x2 列为逆向指标单独处理；标号 y0 取 5 个 0、5 个 1。用 `MLPClassifier(solver="lbfgs", activation="logistic", hidden_layer_sizes=30)` 拟合（计算保持原样），对待判样本作同样标准化后预测类别、各类别概率与训练集准确率。（源程序 ex15_11.py）

In [ ]:
a = np.loadtxt("data/15_11.txt")
x0 = a[:10, :]
x = a[10:, :]
m1 = x0.max(axis=0)
m2 = x0.min(axis=0)
bx0 = (x0 - m2) / (m1 - m2)
bx0[:, 1] = (m1[1] - x0[:, 1]) / (m1[1] - m2[1])
y0 = np.hstack([np.zeros(5), np.ones(5)])
md = MLPClassifier(solver="lbfgs", activation="logistic", hidden_layer_sizes=30).fit(
    bx0, y0
)
bx = (x - m2) / (m1 - m2)
bx[:, 1] = (m1[1] - x[:, 1]) / (m1[1] - m2[1])
yh = md.predict(bx)
print("待判样本类别：,", yh)
print("属于各个类别的概率：\n", md.predict_proba(bx))
print("训练样本的测试准确度：", md.score(bx0, y0))

## 13.12  MLP 回归器预测与拟合曲线

读取 `data/15_12.txt`，前三列为特征 x0、第四列为目标 y0。将 x0 极差标准化并映射到 [-1,1]（`2*(x0-m2)/(m1-m2)-1`）；用 `MLPRegressor(solver="lbfgs", activation="identity", hidden_layer_sizes=10)` 拟合（计算保持原样）。对新输入 x 作同样标准化后预测并打印；再对已知数据回测得相对误差，最后绘制 1990–2009 年原始数据与预测数据的对比曲线。（源程序 ex15_12.py）

In [ ]:
a = np.loadtxt("data/15_12.txt")
x0 = a[:, :3]
y0 = a[:, 3]
m1 = x0.max(axis=0)
m2 = x0.min(axis=0)
bx0 = 2 * (x0 - m2) / (m1 - m2) - 1
md = MLPRegressor(solver="lbfgs", activation="identity", hidden_layer_sizes=10).fit(
    bx0, y0
)
x = np.array([[73.39, 75.55], [3.9635, 4.0975], [0.9880, 1.0268]]).T
bx = 2 * (x - m2) / (m1 - m2) - 1
yh = md.predict(bx)
print("预测值为：,", np.round(yh, 4))
yh0 = md.predict(bx0)
delta = abs(yh0 - y0) / y0 * 100
print("已知数据预测的相对误差：", np.round(delta, 4))
t = np.arange(1990, 2010)
fig, ax = plt.subplots()
ax.plot(t, y0, "--o", label="原始数据")
ax.plot(t, yh0, "-*", label="预测数据")
ax.set_xticks(t)
ax.tick_params(axis="x", labelrotation=55)
ax.legend()
plt.show()